### **AI_AGENT_replyAI**

### **explore les autres fichiers**

In [ ]:
import pandas as pd
import json
import os

# On définit le dossier une seule fois
base_path = r'../The Truman Show - train'

def load_json_robust(filename):
    full_path = os.path.join(base_path, filename)
    try:
        return pd.read_json(full_path)
    except:
        try:
            return pd.read_json(full_path, orient="records")
        except:
            with open(full_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            return pd.json_normalize(data)

# Chargement propre
df_transactions = pd.read_csv(os.path.join(base_path, "transactions.csv"))
df_users        = load_json_robust("users.json")
df_sms          = load_json_robust("sms.json")
df_mails        = load_json_robust("mails.json")
df_locations    = load_json_robust("locations.json")

print("Tous les fichiers sont chargés avec succès !")

In [ ]:
# Affichage simple des 5 premières lignes pour chaque DataFrame
print("--- HEAD TRANSACTIONS ---")
display(df_transactions.head())

print("\n--- HEAD USERS ---")
display(df_users.head())

print("\n--- HEAD SMS ---")
display(df_sms.head())

print("\n--- HEAD MAILS ---")
display(df_mails.head())

print("\n--- HEAD LOCATIONS ---")
display(df_locations.head())

In [ ]:
print("Transactions:", df_transactions.shape)
print("Users:", df_users.shape)
print("SMS:", df_sms.shape)
print("Mails:", df_mails.shape)
print("Locations:", df_locations.shape)

# Crucial - les descriptions des transactions
print(df_transactions['description'].value_counts())

# Les timestamps min et max
print("Période:", df_transactions['timestamp'].min(), 
      "→", df_transactions['timestamp'].max())

# Transactions qui NE sont PAS salary/rent

In [ ]:
# Transactions qui NE sont PAS salary/rent
mask = ~df_transactions['description'].str.contains(
    'Salary|Rent', na=False
)
print("Transactions suspectes potentielles:")
print(df_transactions[mask][['transaction_id','sender_id',
    'recipient_id','amount','description','timestamp']])

# Comparer montants normaux vs suspects
print("\nMontant moyen salary/rent:", 
    df_transactions[~mask]['amount'].mean())
print("Montant moyen autres:", 
    df_transactions[mask]['amount'].mean())

In [ ]:
for i, row in df_mails.iterrows():
    print(f"\n=== MAIL {i} ===")
    print(row['mail'][:300])

In [ ]:
for i, row in df_mails.iterrows():# mail susoect 
    print(f"\n=== MAIL {i} ===")
    print(row['mail'][:500])
    print("---")

In [ ]:
for i, row in df_sms.iterrows():#sms suspect 
    if any(x in row['sms'].lower() for x in 
           ['paypal', 'amazon', 'verify', 'click', 
            'urgent', 'suspend', 'bit.ly', 'paypa1']):
        print(f"\n=== SMS SUSPECT {i} ===")
        print(row['sms'][:300])

In [ ]:
# Corréler SMS phishing avec transactions suspectes
df_transactions['timestamp'] = pd.to_datetime(
    df_transactions['timestamp'])

# Transactions après SMS phishing PayPal (Nov 2087)
paypal_sms_date = pd.to_datetime('2087-11-03')
print("Transactions après phishing PayPal:")
print(df_transactions[
    df_transactions['timestamp'] > paypal_sms_date
][['transaction_id','sender_id','amount',
   'description','timestamp']])

# Transactions après phishing Amazon (Sep 2087)  
amazon_sms_date = pd.to_datetime('2087-09-08')
print("\nTransactions après phishing Amazon:")
print(df_transactions[
    (df_transactions['timestamp'] > amazon_sms_date) &
    (df_transactions['timestamp'] < amazon_sms_date + 
     pd.Timedelta(days=30))
][['transaction_id','sender_id','amount',
   'description','timestamp']])

In [ ]:
# Récapitulatif de tous les signaux suspects détectés
suspects = []

# 1. Description manquante
no_desc = df_transactions[df_transactions['description'].isna()]
print("=== SANS DESCRIPTION ===")
print(no_desc[['transaction_id','sender_id','recipient_id',
               'amount','timestamp']])

# 2. Transactions nocturnes
df_transactions['hour'] = pd.to_datetime(
    df_transactions['timestamp']).dt.hour
night = df_transactions[
    (df_transactions['hour'] >= 0) & 
    (df_transactions['hour'] < 6)
]
print("\n=== NOCTURNES ===")
print(night[['transaction_id','sender_id','amount',
             'description','hour']])

# 3. Recipient inconnu
known_ibans = df_users['iban'].tolist()
unknown = df_transactions[
    ~df_transactions['recipient_iban'].isin(known_ibans)
]
print("\n=== DESTINATAIRES INCONNUS ===")
print(unknown[['transaction_id','recipient_id',
               'amount','description']].head(10))

In [ ]:
# GPS — l'utilisateur était-il physiquement 
# au même endroit que la transaction ?
with open('../The Truman Show - train/locations.json') as f:
    loc = json.load(f)
print(type(loc))
print(len(loc))
print(loc[:2])

In [ ]:
df_locations = pd.DataFrame(loc)
df_locations['timestamp'] = pd.to_datetime(
    df_locations['timestamp'])

# Mapping biotag → user
biotag_map = {
    'RGNR-LNAA-7FF-AUD-0': 'Alain',    # Audincourt
    'GRSC-KRLH-807-DIE-1': 'Karl-Hermann',  # Dietzenbach  
    'BRCH-TRCY-802-HAM-1': 'Tracy'      # Hampstead
}

df_locations['user'] = df_locations['biotag'].map(biotag_map)

# Pour chaque transaction suspecte
# vérifie où était l'utilisateur
suspects_idx = [6, 26, 36, 39, 40, 66]
suspect_txs = df_transactions.iloc[suspects_idx]

for _, tx in suspect_txs.iterrows():
    tx_time = tx['timestamp']
    sender = tx['sender_id']
    
    # Localisation proche de la transaction
    nearby = df_locations[
        (df_locations['biotag'] == sender) &
        (abs(df_locations['timestamp'] - tx_time) < 
         pd.Timedelta(hours=24))
    ]
    
    print(f"\nTX: {tx['transaction_id'][:8]}")
    print(f"Sender: {sender}")
    print(f"Time: {tx_time}")
    print(f"Location nearby: {nearby[['city','lat','lng','timestamp']].to_string()}")

In [ ]:
import pandas as pd
import numpy as np

# ===== CONSTRUCTION FEATURES COMPLÈTES =====

# 1. Features transactions
df = df_transactions.copy()
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour

# Destinataires suspects identifiés
suspicious_recipients = ['MKTSW90199', 'RETRB44116']

df['f_no_description'] = df['description'].isna().astype(int)
df['f_night'] = ((df['hour'] >= 0) & 
                  (df['hour'] < 6)).astype(int)
df['f_unknown_recipient'] = df['recipient_id'].isin(
    suspicious_recipients).astype(int)
df['f_null_recipient'] = df['recipient_id'].isna().astype(int)

# 2. Feature salary (montant > salary mensuel)
salary_map = {
    'RGNR-LNAA-7FF-AUD-0': 34100/12,  # Alain
    'GRSC-KRLH-807-DIE-1': 33200/12,  # Karl
    'BRCH-TRCY-802-HAM-1': 12700/12   # Tracy
}
df['monthly_salary'] = df['sender_id'].map(salary_map)
df['f_amount_suspicious'] = (
    df['amount'] > df['monthly_salary']).astype(int)

# 3. Feature phishing SMS
phishing_dates = {
    'RGNR-LNAA-7FF-AUD-0': [
        pd.Timestamp('2087-11-03'),  # PayPal fake
        pd.Timestamp('2087-09-08'),  # Amazon fake
    ]
}

def check_phishing_before(tx):
    dates = phishing_dates.get(tx['sender_id'], [])
    for d in dates:
        if d <= tx['timestamp'] <= d + pd.Timedelta(days=30):
            return 1
    return 0

df['f_phishing_sms'] = df.apply(check_phishing_before, axis=1)

# 4. Feature GPS absent
df_loc = pd.DataFrame(loc)
df_loc['timestamp'] = pd.to_datetime(df_loc['timestamp'])

def check_gps_absent(tx):
    nearby = df_loc[
        (df_loc['biotag'] == tx['sender_id']) &
        (abs(df_loc['timestamp'] - tx['timestamp']) < 
         pd.Timedelta(hours=24))
    ]
    return 0 if len(nearby) > 0 else 1

df['f_gps_absent'] = df.apply(check_gps_absent, axis=1)

# ===== MATRICE DE CORRÉLATION =====
features = [
    'f_no_description',
    'f_night',
    'f_unknown_recipient', 
    'f_null_recipient',
    'f_amount_suspicious',
    'f_phishing_sms',
    'f_gps_absent'
]

corr_matrix = df[features].corr()

print("=== MATRICE DE CORRÉLATION ===")
print(corr_matrix.round(2).to_string())

print("\n=== TRANSACTIONS SUSPECTES (score total) ===")
df['fraud_score'] = df[features].sum(axis=1)
suspects = df[df['fraud_score'] >= 2].sort_values(
    'fraud_score', ascending=False)
print(suspects[['transaction_id', 'sender_id', 
                 'amount', 'hour', 'fraud_score'] + 
                features].to_string())

In [ ]:
# GPS absent seulement pour les 3 utilisateurs connus
known_users = [
    'RGNR-LNAA-7FF-AUD-0',  # Alain
    'GRSC-KRLH-807-DIE-1',  # Karl
    'BRCH-TRCY-802-HAM-1'   # Tracy
]

def check_gps_absent_v2(tx):
    # Seulement vérifier pour nos 3 utilisateurs
    if tx['sender_id'] not in known_users:
        return 0
    nearby = df_loc[
        (df_loc['biotag'] == tx['sender_id']) &
        (abs(df_loc['timestamp'] - tx['timestamp']) < 
         pd.Timedelta(hours=24))
    ]
    return 0 if len(nearby) > 0 else 1

df['f_gps_absent'] = df.apply(check_gps_absent_v2, axis=1)

# Recalcul
df['fraud_score'] = df[features].sum(axis=1)

print("=== SUSPECTS FINAUX ===")
print(df[df['fraud_score'] >= 1].sort_values(
    'fraud_score', ascending=False)[
    ['transaction_id','sender_id',
     'amount','hour','fraud_score'] + features
].to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Garder seulement les transactions suspectes (score >= 1)
df_suspects = df[df['fraud_score'] >= 1].sort_values(
    'fraud_score', ascending=False).copy()

# Raccourcir les IDs pour lisibilité
df_suspects['tx_short'] = df_suspects['transaction_id'].str[:8]

# Labels plus lisibles
feature_labels = {
    'f_no_description': 'No Description',
    'f_night': 'Night (00-06h)',
    'f_unknown_recipient': 'Unknown Recipient',
    'f_null_recipient': 'Null Recipient',
    'f_amount_suspicious': 'High Amount',
    'f_phishing_sms': 'Phishing SMS',
    'f_gps_absent': 'GPS Absent'
}

# Préparer les données
heatmap_data = df_suspects.set_index('tx_short')[features]
heatmap_data.columns = list(feature_labels.values())

# Score par transaction pour annotation
scores = df_suspects.set_index('tx_short')['fraud_score']

fig, ax = plt.subplots(figsize=(12, 8))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='d',
    cmap='RdYlGn_r',
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='gray',
    ax=ax,
    cbar_kws={
        'label': '0=Normal  1=Suspect',
        'shrink': 0.8
    }
)

# Titre et labels
ax.set_title(
    'Fraud Detection Heatmap — Reply Mirror 2026\n'
    'Red = Suspicious Signal | Green = Normal',
    fontsize=14,
    fontweight='bold',
    pad=20
)
ax.set_xlabel('Fraud Features', fontsize=12)
ax.set_ylabel('Transaction ID', fontsize=12)
ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=30,
    ha='right',
    fontsize=10
)
ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,
    fontsize=9
)

# Ajouter score total à droite
for i, (tx, score) in enumerate(scores.items()):
    ax.text(
        len(features) + 0.3, i + 0.5,
        f'Score: {int(score)}',
        va='center',
        fontsize=9,
        fontweight='bold',
        color='red' if score >= 2 else 'orange'
    )

# Ligne séparatrice entre score 2 et score 1
n_high = len(df_suspects[df_suspects['fraud_score'] >= 2])
ax.axhline(y=n_high, color='red', linewidth=2, 
           linestyle='--', alpha=0.7)
ax.text(
    -0.5, n_high + 0.1,
    '← Certain Fraud | Uncertain →',
    fontsize=8, color='red', style='italic'
)

plt.tight_layout()
plt.savefig('fraud_heatmap.png', dpi=150, 
            bbox_inches='tight')
plt.show()
print("✅ Heatmap sauvegardée : fraud_heatmap.png")

### **agent1** 

### **tools**

In [ ]:
import pandas as pd
import json
from datetime import timedelta

# ===== DONNÉES GLOBALES =====
known_users = {
    'RGNR-LNAA-7FF-AUD-0': {
        'name': 'Alain',
        'salary': 34100/12,
        'city': 'Audincourt'
    },
    'GRSC-KRLH-807-DIE-1': {
        'name': 'Karl',
        'salary': 33200/12,
        'city': 'Dietzenbach'
    },
    'BRCH-TRCY-802-HAM-1': {
        'name': 'Tracy',
        'salary': 12700/12,
        'city': 'Hampstead'
    }
}

suspicious_recipients = ['MKTSW90199', 'RETRB44116']

phishing_keywords = [
    'paypa1', 'amaz0n', 'verify',
    'bit.ly', 'urgent', 'suspend',
    'account lock', 'click here'
]

regular_keywords = [
    'salary', 'rent', 'insurance',
    'mobile', 'internet', 'rideshare',
    'health', 'subscription'
]

# ===== TOOL 1 : TRANSACTION RULES =====
def check_rules(tx: dict) -> dict:
    signals = []
    score = 0

    desc = str(tx.get('description', ''))
    is_regular = any(x in desc.lower() for x in regular_keywords)

    if not tx.get('description') or pd.isna(tx.get('description')):
        score += 0.4
        signals.append("missing_description")

    hour = pd.to_datetime(tx['timestamp']).hour
    if hour < 6:
        score += 0.3
        signals.append("night_transaction")

    if tx.get('recipient_id') in suspicious_recipients:
        score += 0.5
        signals.append("unknown_recipient")

    if not tx.get('recipient_id') or str(tx.get('recipient_id')) == 'nan':
        score += 0.4
        signals.append("null_recipient")

    if float(tx.get('amount', 0)) > 500 and not is_regular:
        score += 0.2
        signals.append("high_amount_suspicious")

    return {
        "score": min(round(score, 2), 1.0),
        "signals": signals,
        "hint": "high_risk" if score >= 0.7
                else "suspicious" if score >= 0.3
                else "low_risk"
    }

# ===== TOOL 2 : PHISHING DETECTOR =====
def check_phishing(tx: dict, sms_list: list, mails_list: list) -> dict:
    sender = tx.get('sender_id', '')

    # Ignorer transactions regulieres
    desc = str(tx.get('description', ''))
    is_regular = any(x in desc.lower() for x in regular_keywords)
    if is_regular:
        return {"score": 0, "signals": [], "hint": "low_risk"}

    sender_phone = {
        'RGNR-LNAA-7FF-AUD-0': '+18151162635',
        'GRSC-KRLH-807-DIE-1': '+49',
        'BRCH-TRCY-802-HAM-1': '+12918009249'
    }

    user_mail_map = {
        'RGNR-LNAA-7FF-AUD-0': 'alain.regnier',
        'GRSC-KRLH-807-DIE-1': 'karl',
        'BRCH-TRCY-802-HAM-1': 'tracy'
    }

    user_phone = sender_phone.get(sender, '')
    user_name = user_mail_map.get(sender, '')
    signals = []
    score = 0
    tx_time = pd.to_datetime(tx['timestamp'])

    for item in sms_list:
        sms_text = str(item) if not isinstance(item, dict) else str(item.get('sms', ''))
        if user_phone not in sms_text:
            continue
        if any(k in sms_text.lower() for k in phishing_keywords):
            try:
                date_str = sms_text.split('Date:')[1].split('\n')[0].strip()
                sms_time = pd.to_datetime(date_str)
                if 0 <= (tx_time - sms_time).days <= 30:
                    score += 0.5
                    signals.append("phishing_sms_before_tx")
                    break
            except:
                continue

    for item in mails_list:
        mail_text = str(item) if not isinstance(item, dict) else str(item.get('mail', ''))
        if user_name not in mail_text.lower():
            continue
        if any(k in mail_text.lower() for k in phishing_keywords):
            score += 0.3
            signals.append("phishing_mail")
            break

    return {
        "score": min(round(score, 2), 1.0),
        "signals": signals,
        "hint": "high_risk" if score >= 0.5
                else "suspicious" if score > 0
                else "low_risk"
    }

# ===== TOOL 3 : GPS ANALYZER =====
def check_gps(tx: dict, locations: list) -> dict:
    signals = []
    score = 0
    sender = tx.get('sender_id', '')

    if sender not in known_users:
        return {"score": 0, "signals": [], "hint": "low_risk"}

    # Ignorer transactions regulieres
    desc = str(tx.get('description', ''))
    is_regular = any(x in desc.lower() for x in regular_keywords)
    if is_regular:
        return {"score": 0, "signals": [], "hint": "low_risk"}

    tx_time = pd.to_datetime(tx['timestamp'])
    df_loc = pd.DataFrame(locations)
    df_loc['timestamp'] = pd.to_datetime(df_loc['timestamp'])

    diff = (df_loc['timestamp'] - tx_time).abs()
    nearby = df_loc[
        (df_loc['biotag'] == sender) &
        (diff < pd.Timedelta(hours=24))
    ]

    if len(nearby) == 0:
        score += 0.3
        signals.append("no_gps_near_transaction")
    else:
        user_city = known_users[sender]['city']
        cities = nearby['city'].unique()
        if user_city not in cities:
            score += 0.4
            signals.append("gps_city_mismatch")

    return {
        "score": min(round(score, 2), 1.0),
        "signals": signals,
        "hint": "high_risk" if score >= 0.4
                else "suspicious" if score > 0
                else "low_risk"
    }

# ===== TOOL 4 : USER PROFILER =====
def check_user_profile(tx: dict) -> dict:
    signals = []
    score = 0
    sender = tx.get('sender_id', '')

    if sender not in known_users:
        return {"score": 0, "signals": [], "hint": "low_risk"}

    user = known_users[sender]
    amount = float(tx.get('amount', 0))
    desc = str(tx.get('description', ''))

    is_regular = any(x in desc.lower() for x in regular_keywords)
    if is_regular:
        return {"score": 0, "signals": [], "hint": "low_risk"}

    if amount > user['salary']:
        score += 0.4
        signals.append(f"amount_exceeds_monthly_salary_{round(user['salary'])}EUR")

    if amount > user['salary'] * 2:
        score += 0.3
        signals.append("amount_exceeds_2x_salary")

    return {
        "score": min(round(score, 2), 1.0),
        "signals": signals,
        "hint": "high_risk" if score >= 0.5
                else "suspicious" if score > 0
                else "low_risk"
    }

# ===== SCORING GLOBAL =====
def score_transaction(tx: dict, sms_list: list,
                      mails_list: list, locations: list) -> dict:

    r1 = check_rules(tx)
    r2 = check_phishing(tx, sms_list, mails_list)
    r3 = check_gps(tx, locations)
    r4 = check_user_profile(tx)

    all_signals = (r1['signals'] + r2['signals'] +
                   r3['signals'] + r4['signals'])

    MAX_SIGNALS = 10
    normalized_score = round(len(all_signals) / MAX_SIGNALS, 2)

    if normalized_score >= 0.4:
        hint = "high_risk"
    elif normalized_score >= 0.1:
        hint = "suspicious"
    else:
        hint = "low_risk"

    amount = float(tx.get('amount', 0))
    desc = str(tx.get('description', ''))
    is_regular = any(x in desc.lower() for x in regular_keywords)

    if amount > 1000 and normalized_score >= 0.1 and not is_regular:
        hint = "high_risk"

    return {
        "combined_score": normalized_score,
        "signal_count": len(all_signals),
        "signals": all_signals,
        "hint": hint,
        "amount": amount,
        "details": {
            "rules": r1,
            "phishing": r2,
            "gps": r3,
            "profile": r4
        }
    }


In [ ]:
# Scorer toutes les transactions
results = []

for _, tx in df_transactions.iterrows():
    tx_dict = tx.to_dict()
    score = score_transaction(
        tx_dict, sms_list, mails_list, loc)
    results.append({
        'transaction_id': tx_dict['transaction_id'],
        'amount': tx_dict['amount'],
        'score': score['combined_score'],
        'signal_count': score['signal_count'],
        'signals': score['signals'],
        'hint': score['hint']
    })

df_results = pd.DataFrame(results)
print(df_results[df_results['score'] > 0].sort_values(
    'score', ascending=False).to_string())

### **agent1 et 2**

In [ ]:
import os
import json
import pandas as pd
import ulid
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langfuse import Langfuse, observe
from langfuse.langchain import CallbackHandler

load_dotenv()

# ===== CONFIG =====
model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="anthropic/claude-haiku-4-5",
    temperature=0,
    max_tokens=50
)

model_deep = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="anthropic/claude-sonnet-4",
    temperature=0,
    max_tokens=200
)

langfuse_client = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_HOST")
)

# ===== AGENT 1 : LLM DECISION =====
@observe()
def agent1_decide(tx: dict,
                  score_result: dict,
                  session_id: str) -> dict:

    handler = CallbackHandler()

    prompt = f"""You are a fraud detection agent.

Transaction:
- type: {tx.get('transaction_type')}
- amount: {tx.get('amount')}EUR
- hour: {pd.to_datetime(tx['timestamp']).hour}h
- description: {tx.get('description')}
- recipient: {tx.get('recipient_id')}

Risk analysis:
- score: {score_result['combined_score']}
- signals: {score_result['signals']}
- hint: {score_result['hint']}

IMPORTANT: Flag as fraud if ANY signal is present.
Return ONLY JSON:
{{"fraud": 0 or 1, "reason": "short reason"}}"""

    response = model.invoke(
        [HumanMessage(content=prompt)],
        config={
            "callbacks": [handler],
            "metadata": {"langfuse_session_id": session_id}
        }
    )

    try:
        text = response.content.strip()
        text = text.replace("```json", "").replace("```", "")
        result = json.loads(text)
        return {
            "fraud": int(result.get("fraud", 0)),
            "reason": result.get("reason", "")
        }
    except:
        fraud = 1 if score_result['combined_score'] >= 0.1 else 0
        return {"fraud": fraud, "reason": "parse_error"}

# ===== RUN AGENT 1 =====
def run_agent1(folder: str, session_id: str) -> list:

    df_tx = pd.read_csv(f'{folder}/transactions.csv')

    with open(f'{folder}/sms.json') as f:
        sms_raw = json.load(f)
    with open(f'{folder}/mails.json') as f:
        mails_raw = json.load(f)
    with open(f'{folder}/locations.json') as f:
        locations = json.load(f)

    sms_list = [{'sms': s} if isinstance(s, str) else s for s in sms_raw]
    mails_list = [{'mail': m} if isinstance(m, str) else m for m in mails_raw]

    suspects = []

    for _, tx in df_tx.iterrows():
        tx_dict = tx.to_dict()

        score_result = score_transaction(
            tx_dict, sms_list, mails_list, locations)

        decision = agent1_decide(
            tx_dict, score_result, session_id)

        if decision['fraud'] == 1:
            suspects.append({
                'transaction_id': tx_dict['transaction_id'],
                'amount': tx_dict['amount'],
                'score': score_result['combined_score'],
                'signals': score_result['signals'],
                'reason': decision['reason']
            })

    langfuse_client.flush()
    print(f"✅ Agent1 → {len(suspects)} suspects détectés")
    for s in suspects:
        print(f"  → {s['transaction_id'][:8]} | "
              f"{s['amount']}€ | {s['reason']}")

    return suspects

# ===== AGENT 2 : DEEP REVIEWER =====
@observe()
def agent2_review(suspect: dict,
                  tx_full: dict,
                  user_info: dict,
                  session_id: str) -> dict:

    handler = CallbackHandler()

    prompt = f"""You are a senior fraud investigator.

Agent1 flagged this transaction as suspicious.
Your job: confirm or reject this fraud flag.

Transaction details:
- type: {tx_full.get('transaction_type')}
- amount: {tx_full.get('amount')}EUR
- hour: {pd.to_datetime(tx_full['timestamp']).hour}h
- description: {tx_full.get('description')}
- recipient: {tx_full.get('recipient_id')}
- balance_after: {tx_full.get('balance_after')}

User profile:
- name: {user_info.get('name')}
- monthly_salary: {round(user_info.get('salary', 0))}EUR
- city: {user_info.get('city')}

Agent1 signals: {suspect['signals']}
Agent1 reason: {suspect['reason']}
Agent1 score: {suspect['score']}

Be strict. High amount fraud costs more.
Return ONLY JSON:
{{"fraud": 0 or 1, "confidence": 0.0-1.0, "reason": "detailed reason"}}"""

    response = model_deep.invoke(
        [HumanMessage(content=prompt)],
        config={
            "callbacks": [handler],
            "metadata": {"langfuse_session_id": session_id}
        }
    )

    try:
        text = response.content.strip()
        text = text.replace("```json", "").replace("```", "")
        result = json.loads(text)
        return {
            "fraud": int(result.get("fraud", 0)),
            "confidence": float(result.get("confidence", 0.5)),
            "reason": result.get("reason", "")
        }
    except:
        return {"fraud": 1, "confidence": 0.5, "reason": "parse_error"}

# ===== RUN AGENT 2 =====
def run_agent2(suspects: list, folder: str, session_id: str) -> list:

    df_tx = pd.read_csv(f'{folder}/transactions.csv')

    known_users_local = {
        'RGNR-LNAA-7FF-AUD-0': {
            'name': 'Alain', 'salary': 34100/12, 'city': 'Audincourt'},
        'GRSC-KRLH-807-DIE-1': {
            'name': 'Karl', 'salary': 33200/12, 'city': 'Dietzenbach'},
        'BRCH-TRCY-802-HAM-1': {
            'name': 'Tracy', 'salary': 12700/12, 'city': 'Hampstead'}
    }

    confirmed_frauds = []

    for suspect in suspects:
        tx_row = df_tx[
            df_tx['transaction_id'] == suspect['transaction_id']]
        if tx_row.empty:
            continue

        tx_full = tx_row.iloc[0].to_dict()
        sender = tx_full.get('sender_id', '')
        user_info = known_users_local.get(sender, {
            'name': 'unknown', 'salary': 0, 'city': 'unknown'})

        review = agent2_review(
            suspect, tx_full, user_info, session_id)

        if review['fraud'] == 1 or review['confidence'] >= 0.5:
            confirmed_frauds.append({
                'transaction_id': suspect['transaction_id'],
                'amount': suspect['amount'],
                'confidence': review['confidence'],
                'reason': review['reason']
            })

    langfuse_client.flush()
    print(f"✅ Agent2 → {len(confirmed_frauds)} fraudes confirmées")
    for f in confirmed_frauds:
        print(f"  → {f['transaction_id'][:8]} | {f['amount']}€ | "
              f"conf:{f['confidence']} | {f['reason'][:50]}")

    return confirmed_frauds

# ===== MAIN =====
def run_full_pipeline(folder: str) -> str:
    session_id = f"NETIZEN-{ulid.new().str}"
    print(f"🚀 Session ID: {session_id}")
    print(f"📁 Dataset: {folder}\n")

    print("=== AGENT 1 : FAST SCREENER ===")
    suspects = run_agent1(folder, session_id)

    print(f"\n=== AGENT 2 : DEEP REVIEWER ===")
    confirmed = run_agent2(suspects, folder, session_id)

    output_file = 'output_train.txt'
    with open(output_file, 'w', encoding='utf-8') as f:
        for fraud in confirmed:
            f.write(fraud['transaction_id'] + '\n')

    print(f"\n✅ Output sauvegardé : {output_file}")
    print(f"📊 {len(confirmed)} fraudes détectées")
    print(f"🔑 Session ID : {session_id}")

    return session_id

# LANCER ICI
session_id = run_full_pipeline("../The Truman Show - validation")

In [ ]:
import pandas as pd
import json

folder = "../Brave New World - train"

df_tx = pd.read_csv(f'{folder}/transactions.csv')
print("Shape:", df_tx.shape)
print(df_tx.head(3))

with open(f'{folder}/users.json') as f:
    users = json.load(f)
print("\nUsers:", list(users.items())[:2])